<a href="https://colab.research.google.com/github/pranjal7398/AI_using_Pytorch/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Chunking**

In [ ]:
# chunking - divide the large data in samller chunks

text='''Hey my name is pranjal.
I am from varanasi.
I am currently pursuing mtech from nit dgp in AI & DS.
I have completed my btech in computer science and engineering .
'''
print(text)

#chunking
chunks = text.strip().split("\n")
print(chunks)

print()
print('The proper list is:- ')

# to see these chunks properly
for chunk in chunks:
  print(chunk)

In [1]:
# Multiple chunking
text='''Hey my name is pranjal.
I am from varanasi.
I am currently pursuing mtech from nit dgp in AI & DS.
I have completed my btech in computer science and engineering .
'''

lines = text.strip().split("\n")
chunks = []
join_chunk = [] # this is a clean chunk text

for i in range(0, len(lines) , 2): #we are merging 2 chunks and creating 1 chunk
  chunk = lines[i:i+2]
  chunks.append(chunk)
  j_chunk = " ".join(lines[i:i+2])
  join_chunk.append(j_chunk)

print("Before joining :- ")  #==> returns a list of chunk
for chunk in chunks:
  print(chunk)

print()

print("After joining :- ") #==> returns a text of chunks
for j_chunk in join_chunk:
  print(j_chunk)

Before joining :- 
['Hey my name is pranjal.', 'I am from varanasi.']
['I am currently pursuing mtech from nit dgp in AI & DS.', 'I have completed my btech in computer science and engineering .']

After joining :- 
Hey my name is pranjal. I am from varanasi.
I am currently pursuing mtech from nit dgp in AI & DS. I have completed my btech in computer science and engineering .


**Embedding**

we are telling to the model that convert the below data in embedded form i.e. numerical values


In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

result = client.models.embed_content(
    model = 'gemini-embedding-2',
    contents = 'Refund requests are allowed within 7 days'
)

print(result)

In [10]:
# comparing 2 sentences and finding similarity score between them
import numpy as np
from google import genai
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

text1 = 'Refund requests are allowed within 7 days'
text2 = 'Can i get my money back ater 5 days'

result1 = client.models.embed_content(
    model = 'gemini-embedding-2',
    contents = text1
)
result2 = client.models.embed_content(
    model = 'gemini-embedding-2',
    contents = text2
)

vector1 = result1.embeddings[0].values
vector2 = result2.embeddings[0].values
print('length of vector 1 = ' ,len(vector1))
print('length of vector 2 = ' ,len(vector2))

def cosine_similarity(a,b):
  a = np.array(a)
  b = np.array(b)

  return np.dot(a,b)/(
      np.linalg.norm(a) * np.linalg.norm(b)
  )

score = cosine_similarity(vector1,vector2)
print('similarity score is - ',score)


length of vector 1 =  3072
length of vector 2 =  3072
similarity score is -  0.8283201536747469


In [12]:
import numpy as np
from google import genai
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

documents = [
    'Refund requests are allowed within 7 days',
    'Classes of python will run from monday to friday',
    'Students will get 3 interview oppurtunities'
]

question = 'can i get my money back after 5 days'

document_vector = []
for document in documents :
  result = client.models.embed_content(
    model = 'gemini-embedding-2',
    contents = document
    )

  vector = result.embeddings[0].values
  document_vector.append(vector)

question_result = client.models.embed_content(
    model = 'gemini-embedding-2',
    contents = question
    )
question_vector = question_result.embeddings[0].values

def cosine_similarity(a,b):
  a = np.array(a)
  b = np.array(b)

  return np.dot(a,b)/(
      np.linalg.norm(a) * np.linalg.norm(b)
  )
for document,vector in zip(documents,document_vector):
  score = cosine_similarity(
      question_vector,
      vector
  )
  print(round(score,3) , document)

0.783 Refund requests are allowed within 7 days
0.573 Classes of python will run from monday to friday
0.526 Students will get 3 interview oppurtunities


**Till now i have learnt Document -> Chunking -> Embedding -> Vector** <br>
***Now we are going to learn Retrieval***

In [16]:
import numpy as np
from google import genai
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

documents = [
    'Refund requests are allowed within 7 days',
    'Classes of python will run from monday to friday',
    'Students will get 3 interview oppurtunities',
    'Course is available for 6 months'
]

question = 'How long is the course'

document_vector = []
for document in documents :
  result = client.models.embed_content(
    model = 'gemini-embedding-2',
    contents = document
    )

  vector = result.embeddings[0].values
  document_vector.append(vector)

question_result = client.models.embed_content(
    model = 'gemini-embedding-2',
    contents = question
    )
question_vector = question_result.embeddings[0].values

def cosine_similarity(a,b):
  a = np.array(a)
  b = np.array(b)

  return np.dot(a,b)/(
      np.linalg.norm(a) * np.linalg.norm(b)
  )

scores = []
for document, vector in zip(documents, document_vector):
  score = cosine_similarity(
      question_vector,
      vector
  )
  print(round(score,3) , document)
  scores.append(score)

# we are comparing the question to the statements present in the document and the nearly matching best score is choosen i.e. the highest score

best_index = np.argmax(scores)
best_doc = documents[best_index]

print(best_doc)

0.6 Refund requests are allowed within 7 days
0.616 Classes of python will run from monday to friday
0.621 Students will get 3 interview oppurtunities
0.75 Course is available for 6 months
Course is available for 6 months
